In [0]:



from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import pipelines as dp

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")

SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

In [0]:
df = spark.read.table(SILVER)

In [0]:
@dp.materialized_view(name="fact_vehicle_status")
def fact_vehicle_status():
    dim_vehicle = spark.read.table("dim_vehicle")
    dim_route = spark.read.table("dim_route")
    dim_destination = spark.read.table("dim_destination")
    dim_time = spark.read.table("dim_time")

    return(
        df
        .withColumn("date", F.to_date("event_time_local"))
        .withColumn("hour", F.hour("event_time_local"))
        .join(dim_vehicle.select("vehicleId", "vehicle_key"), on="vehicleId", how="left")
        .join(dim_route.select("routeId", "route_key"), on="routeId", how="left")
        .join(dim_destination.select("headsign", "destination_key"), on="headsign", how="left")
        .join(dim_time.select("date", "hour", "time_key"), on =["date", "hour"], how = "left")
        .select("vehicle_key", "route_key", "destination_key", "time_key" "tripId", "event_time_local", "speed", "delay", "delay_min", "lat", "lon", "gpsQuality", "gps_ok", "has_trip", "is_delayed", "is_stopped", "is_moving", "delay_bucket")
    )
